In [21]:
import json
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "agents").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.ontology_qa_agent import build_ontology_qa_agent


In [22]:
DATA_PATH = PROJECT_ROOT / "data" / "test_questions_v1_0_manual.json"
samples = json.loads(DATA_PATH.read_text(encoding="utf-8"))

sample_index = 3
sample = samples[sample_index]

print(sample["question"])
print(json.dumps(sample["options"], ensure_ascii=False, indent=2))

Ai là vợ của Barack Obama?
[
  {
    "id": 1,
    "text": "Ông chưa có vợ"
  },
  {
    "id": 2,
    "text": "Eva Obama"
  },
  {
    "id": 3,
    "text": "Sasha Oboma"
  },
  {
    "id": 4,
    "text": "Michelle Obama"
  },
  {
    "id": 5,
    "text": "Malia Obama"
  }
]


In [23]:
graph = build_ontology_qa_agent(max_iterations=15)

In [24]:
graph_input = {
    "question": sample["question"],
    "options": sample["options"],
}

final_output = None
total_input_tokens = 0
total_output_tokens = 0

start_time = time.perf_counter()

for event in graph.stream(graph_input, stream_mode="updates"):
    for node_name, update in event.items():
        print(f"\n{'=' * 20} {node_name} {'=' * 20}")

        if node_name == "agent":
            message = update["messages"][-1]
            print("content:", message.content)
            print("tool_calls:", json.dumps(message.tool_calls, ensure_ascii=False, indent=2, default=str))

            # Token usage từ OpenRouter (qua LangChain response_metadata)
            token_usage = message.response_metadata.get("token_usage", {})
            input_tokens = token_usage.get("prompt_tokens", 0) or 0
            output_tokens = token_usage.get("completion_tokens", 0) or 0
            total_input_tokens += input_tokens
            total_output_tokens += output_tokens
            if input_tokens or output_tokens:
                print(f"tokens: input={input_tokens}, output={output_tokens}")

        elif node_name == "tools":
            for message in update.get("messages", []):
                print(f"{message.name}: {message.content}")
        elif node_name == "post_tool":
            print("generated_sparqls:")
            for query in update.get("generated_sparqls", []):
                print(repr(query))
            print("executions:", json.dumps(update.get("executions", []), ensure_ascii=False, indent=2, default=str))
        elif node_name == "initialize":
            print("initialized keys:", sorted(update))
        elif node_name == "retry":
            print(update["messages"][-1].content)
        else:
            print(json.dumps(update, ensure_ascii=False, indent=2, default=str))

        if node_name == "finalize":
            final_output = {
                "selected_option_id": update.get("selected_option_id"),
                "answer": update.get("answer"),
                "result": update.get("result"),
            }

elapsed_ms = (time.perf_counter() - start_time) * 1000

print("\nFINAL OUTPUT")
print(json.dumps(final_output, ensure_ascii=False, indent=2, default=str))
print(f"\nTOKENS  input={total_input_tokens}  output={total_output_tokens}  total={total_input_tokens + total_output_tokens}")
print(f"TIME    {elapsed_ms:.0f} ms")



==================== initialize ====================
initialized keys: ['actions', 'executions', 'generated_sparqls', 'iteration_count', 'messages', 'steps']

==================== agent ====================
content: 
tool_calls: [
  {
    "name": "search_entity_by_label",
    "args": {
      "query": "Barack Obama"
    },
    "id": "chatcmpl-tool-a49220f1a689236f",
    "type": "tool_call"
  }
]
tokens: input=3282, output=21

==================== tools ====================
search_entity_by_label: [{"label": "Barack Obama", "uri": "http://dbpedia.org/resource/Barack_Obama", "score": 102.32973, "types": ["http://dbpedia.org/ontology/Politician"], "type_labels": ["Politician"]}, {"label": "Barack Obama Academy", "uri": "http://dbpedia.org/resource/Barack_Obama_Academy", "score": 90.74271, "types": ["http://dbpedia.org/ontology/School"], "type_labels": ["School"]}, {"label": "Barack Obama Day", "uri": "http://dbpedia.org/resource/Barack_Obama_Day", "score": 90.74271, "types": ["http://dbpe